<a href="https://colab.research.google.com/github/arullka/alg_descrmath_evstigneeva.a.n/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Лабораторная работа №3 "Задачи NLP"
Фамилия, имя: Евстигнеева Арина

In [ ]:
#Подключим все необходимые библиотеки
!pip install -q gdown transformers nltk

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import nltk
from tqdm import tqdm_notebook
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
from typing import List
import string

import seaborn
seaborn.set(palette='summer')
nltk.download('punkt')

import os
import random
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import matplotlib.pyplot as plt
from IPython.display import clear_output


device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


'cuda'

In [ ]:
Задание 1

In [ ]:
#Загрузим данные
!gdown --folder https://drive.google.com/drive/folders/1epWf0Wwixxv9juicjdCYYVz351vEe9yM?usp=drive_link

Retrieving folder contents
Processing file 1U9L5hXUDWIBdzvTWCyi-UBxHsfbqBmPH test_authors.csv
Processing file 1zfjcjc0S_9DQWZ1MCftKdF9iPct89vFx train_authors.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1U9L5hXUDWIBdzvTWCyi-UBxHsfbqBmPH
To: /content/Text_Author/test_authors.csv
100% 962k/962k [00:00<00:00, 12.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zfjcjc0S_9DQWZ1MCftKdF9iPct89vFx
To: /content/Text_Author/train_authors.csv
100% 5.16M/5.16M [00:00<00:00, 34.6MB/s]
Download completed


In [ ]:
#Посмотрим на данные из датасета
train_data = pd.read_csv('Text_Author/train_authors.csv')
test_data  = pd.read_csv('Text_Author/test_authors.csv')
train_data.head()

,text,author
0,"Студент, который все это нам рассказал, прилет...",Bulychev
1,"-Что не укладывается в голове,- произнес отец ...",Pratchett
2,"-Ш-ш-ш,- сказал я.- Спи.\n-Не могу,- ответила ...",King
3,-В Севастополь? Новое задание имеет отношение ...,Akunin
4,"-Ты прав,- сказал Сева.- Даже если это космиче...",Bulychev


In [ ]:
#Назначим каждому автору идентификатор
writers = ['Akunin', 'Bulychev', 'Chehov', 'Dostoevsky',
           'Gogol', 'King', 'Pratchett', 'Remark']
writers_to_label = {w:i for i,w in enumerate(writers)}
label_to_writers = {i:w for i,w in enumerate(writers)}

#Подготовим данные для задачи классификации текстов по авторам, преобразую сырые данные в словари tarain и test
dataset = {}
dataset['train'] = [
    {'text':t, 'author':writers_to_label[a]}
    for t,a in zip(train_data['text'], train_data['author'])
]

dataset['test'] = [
    {'text':t, 'author':0}
    for t in test_data['text']
]

In [ ]:
#Токенизируем
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

nltk.download('punkt_tab')
#Считаем частоты слов
counter = Counter()
for x in dataset['train']:
    counter.update(tokenize(x['text']))

#Создаем словарь слов
vocab = {w:i+2 for i,(w,_) in enumerate(counter.most_common(20000))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

#Создаем класс Dataset
class RNNDataset(Dataset):
    def __init__(self, data, vocab, max_len=200):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = tokenize(text)[:self.max_len]
        ids = [self.vocab.get(t,1) for t in tokens]
        ids += [0]*(self.max_len-len(ids))
        return ids

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(self.encode(item['text'])),
            'label': torch.tensor(item['author'])
        }

train_loader = DataLoader(RNNDataset(dataset['train'], vocab),
                          batch_size=64, shuffle=True)

test_dataloader = DataLoader(RNNDataset(dataset['test'], vocab),
                            batch_size=64)

#Создаем модель LSTM
class RNNModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.lstm = nn.LSTM(128, 128, num_layers=2,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128*4, 8)

    def forward(self, x):
        x = self.emb(x)
        out,_ = self.lstm(x)

        out = torch.cat([out.mean(1), out.max(1)[0]], dim=1)
        return self.fc(out)

#Обучаем модель
model = RNNModel(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for batch in train_loader:
        x = batch['input_ids'].to(device)
        y = batch['label'].to(device)

        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

#Делаем предсказания
def get_predictions(model, dataloader):
    model.eval()
    preds = []
    with torch.no_grad():
        for b in dataloader:
            x = b['input_ids'].to(device)
            preds.append(model(x).argmax(1))
    return torch.cat(preds).cpu().numpy()

predictions = get_predictions(model, test_dataloader)
predictions = [label_to_writers[x] for x in predictions]


#Сохраним результат
np.save('submission_rnn03.npy', predictions, allow_pickle=True)


#Проверяем то, что модель обучилась. Можно менять параметр i
i = 10
sample = dataset['train'][i]['text']
true_label = dataset['train'][i]['author']

pred = model(
    torch.tensor(RNNDataset(dataset['train'], vocab).encode(sample)).unsqueeze(0).to(device)
).argmax(1).item()

print("Pravda:", label_to_writers[true_label])
print("Predskazanie:", label_to_writers[pred])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Pravda: Akunin
Predskazanie: Akunin


In [ ]:
Задание 2

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/151O9W-yp6TMP0Wmrm_wzbu8F7sq4cm_4?usp=sharing

Retrieving folder contents
Processing file 1ifRDFC5b9V08XzWo1my7XCCGh_ji3jt1 holdout_texts.npy
Processing file 1k-dMMYNPqOjeppDoTWd7nR3Ln2VI0ZVv STT2_train_task.tsv.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1ifRDFC5b9V08XzWo1my7XCCGh_ji3jt1
To: /content/SST2/holdout_texts.npy
100% 45.4k/45.4k [00:00<00:00, 53.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1k-dMMYNPqOjeppDoTWd7nR3Ln2VI0ZVv
To: /content/SST2/STT2_train_task.tsv.txt
100% 720k/720k [00:00<00:00, 8.81MB/s]
Download completed


In [ ]:
#Преобразуем holdout-выборку в numpy.array
texts_holdout = np.load('/content/SST2/holdout_texts.npy', allow_pickle=True)
texts_holdout[:5]

#Загрузим обучающие и тестовые размеченные данные
df = pd.read_csv(
    '/content/SST2/STT2_train_task.tsv.txt',
    delimiter='\t',
    header=None
)

texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]

texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]

In [ ]:
#Получаем эмбеддинги
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert.eval()

def get_bert_embeddings(texts, model, tokenizer, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(list(batch_texts), padding=True, truncation=True,
                        return_tensors='pt').to(device)

        with torch.no_grad():
            out = model(**enc).last_hidden_state[:,0,:]
        embeddings.append(out.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

#Превращаем текст в числа и обучаем классификатор
X_train = get_bert_embeddings(texts_train, bert, tokenizer)
X_test  = get_bert_embeddings(texts_test, bert, tokenizer)
X_hold  = get_bert_embeddings(texts_holdout, bert, tokenizer)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

print("ACC:", accuracy_score(y_test, clf.predict(X_test)))


#Сохраним результат
out_dict = {
    'train': clf.predict_proba(X_train),
    'test': clf.predict_proba(X_test),
    'holdout': clf.predict_proba(X_hold)
}

np.save('submission_bert03.npy', out_dict, allow_pickle=True)

#Проверяем качество
from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc) #должно быть больше 80%

#Проверяем вероятности
probs = clf.predict_proba(X_test)

print("Shape:", probs.shape)  # должно быть (1920, 2)
print("Пример:", probs[0])
print("Сумма:", probs[0].sum())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ACC: 0.8447916666666667
Accuracy: 0.8447916666666667
shape: (1920, 2)
пример: [0.99212119 0.00787881]
сумма: 1.0


In [ ]:
Задание 3

In [ ]:
#Подготовим данные
decoder_hidden_state = np.array([7, 11, 4]).astype(float)[:, None]

encoder_hidden_states = np.array([
    [1, 5, 11, 4, -4],
    [7, 4, 1, 2, 2],
    [8, 12, 2, 11, 5],
    [-9, 0, 1, 8, 12]
]).astype(float).T

W_mult = np.array([
    [-0.78, -0.97, -1.09, -1.79, 0.24],
    [0.04, -0.27, -0.98, -0.49, 0.52],
    [1.08, 0.91, -0.99, 2.04, -0.15]
])

In [ ]:
import numpy as np
from scipy.special import softmax

#Реализуем мультипликативное внимание
def multiplicative_attention(decoder_hidden_state, encoder_hidden_states, W_mult):

    weighted_encoder_states = np.dot(W_mult, encoder_hidden_states)
    attention_scores = np.dot(decoder_hidden_state.T, weighted_encoder_states)
    softmax_vector = softmax(attention_scores)
    attention_vector = softmax_vector.dot(encoder_hidden_states.T).T

    return attention_vector

#Сохраним результат
np.save('nlp02.npy', attention_vector, allow_pickle=True)

#Проведем проверку
attention_vec_mult = multiplicative_attention(decoder_hidden_state, encoder_hidden_states, W_mult)
print(attention_vec_mult)


[[-9.00000000e+00]
 [ 1.01553049e-19]
 [ 1.00000000e+00]
 [ 8.00000000e+00]
 [ 1.20000000e+01]]
